Cloud-only data dependency: this notebook reads the label Parquets written by `notebooks/nepc.ipynb` and `notebooks/avpc.ipynb`. It makes no LLM calls.

# Primary Subtype Labels (NEPC > AVPC > conventional)

Merges the full `nepc_labels.parquet` and `avpc_labels.parquet` tables (one row per patient,
every column from both) and adds one column, `primary_label`, assigned in precedence order:

1. `nepc` — `nepc_status == "nepc"`.
2. `avpc` — not NEPC, and `is_avpc` is true.
3. `conventional` — `nepc_status == "non_nepc"` and `is_avpc` is false.

`primary_label` is null when a higher-precedence label is unresolved: the NEPC or AVPC
extraction failed or never ran for that patient (the pipeline left its value null), or the
patient is missing from one of the two files. A NEPC-positive patient is always `nepc`.

The two column names both files share get a pipeline prefix: `nepc_review_status` /
`avpc_review_status` and `nepc_gate_rejections` / `avpc_gate_rejections`. AVPC criteria use the
paper's numbering (C2 = exclusively visceral, C3 = lytic). Re-run this notebook after re-running
either pipeline (including `REBUILD_LABELS_ONLY`).

In [ ]:
from pathlib import Path
import os
import sys

import polars as pl
from IPython.display import display


def find_repo_root(start):
    """Find the repo root (contains pyproject.toml) from the notebook directory."""
    start = Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Could not locate the repo root (pyproject.toml)")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from preprocessing.parquet_io import write_parquet_atomic

DEFAULT_DATA_ROOT = Path(
    os.environ.get("LLM_ANNOTATIONS_DATA_PATH", "/data/gusev/USERS/jpconnor/data/LLM_annotations/")
)
pl.Config.set_fmt_str_lengths(120)

print(f"Repo root: {REPO_ROOT}")

In [ ]:
# Inputs: the OUTPUT_DIRs of notebooks/nepc.ipynb and notebooks/avpc.ipynb.
NEPC_LABELS_PATH = DEFAULT_DATA_ROOT / "LLM_nepc" / "nepc_labels.parquet"
AVPC_LABELS_PATH = DEFAULT_DATA_ROOT / "LLM_avpc" / "avpc_labels.parquet"

OUTPUT_DIR = DEFAULT_DATA_ROOT / "LLM_primary_subtype"
OUTPUT_PATH = OUTPUT_DIR / "primary_subtype_labels.parquet"

# Writing only replaces the merged file; neither pipeline's outputs are touched.
WRITE_LABELS = True

for path in (NEPC_LABELS_PATH, AVPC_LABELS_PATH):
    print(f"{Path(path).exists()}: {path}")

In [ ]:
missing = [str(p) for p in (NEPC_LABELS_PATH, AVPC_LABELS_PATH) if not Path(p).exists()]
if missing:
    raise FileNotFoundError(f"Run the NEPC/AVPC notebooks first; missing: {missing}")

SHARED_COLUMNS = ("review_status", "gate_rejections")
nepc = pl.read_parquet(NEPC_LABELS_PATH).rename({c: f"nepc_{c}" for c in SHARED_COLUMNS})
avpc = pl.read_parquet(AVPC_LABELS_PATH).rename({c: f"avpc_{c}" for c in SHARED_COLUMNS})
for name, frame in (("NEPC", nepc), ("AVPC", avpc)):
    if frame["DFCI_MRN"].is_duplicated().any():
        raise ValueError(f"{name} labels have duplicate DFCI_MRN rows")
overlap = (set(nepc.columns) & set(avpc.columns)) - {"DFCI_MRN"}
if overlap:
    raise ValueError(f"Unexpected shared columns between the label files: {sorted(overlap)}")

nepc_mrns, avpc_mrns = set(nepc["DFCI_MRN"]), set(avpc["DFCI_MRN"])
print(f"NEPC labels: {nepc.height} patients, {nepc.width} columns; "
      f"AVPC labels: {avpc.height} patients, {avpc.width} columns")
print(f"In both: {len(nepc_mrns & avpc_mrns)}; NEPC only: {len(nepc_mrns - avpc_mrns)}; "
      f"AVPC only: {len(avpc_mrns - nepc_mrns)}")
if nepc_mrns != avpc_mrns:
    print("WARNING: the two runs used different cohorts. Patients missing from one file "
          "get a null primary_label unless NEPC is positive.")

In [ ]:
non_nepc = pl.col("nepc_status") == "non_nepc"
# Null comparisons are null, which `when` treats as false: unresolved values fall through.
primary_label = (
    pl.when(pl.col("nepc_status") == "nepc").then(pl.lit("nepc"))
    .when(non_nepc & pl.col("is_avpc")).then(pl.lit("avpc"))
    .when(non_nepc & pl.col("is_avpc").not_()).then(pl.lit("conventional"))
    .otherwise(pl.lit(None, dtype=pl.Utf8))
)

merged = (
    nepc.join(avpc, on="DFCI_MRN", how="full", coalesce=True)
    .with_columns(primary_label.alias("primary_label"))
    .select("DFCI_MRN", "primary_label", pl.exclude("DFCI_MRN", "primary_label"))
    .sort("DFCI_MRN")
)
print(f"Merged: {merged.height} patients, {merged.width} columns")

In [ ]:
print(merged["primary_label"].value_counts().sort("count", descending=True))

# NEPC x AVPC overlap: NEPC patients who also meet AVPC are labelled nepc.
print(merged.group_by("nepc_status", "is_avpc").len().sort("nepc_status", "is_avpc", nulls_last=True))

# Why primary_label is null: the review status of each pipeline for those patients.
print(merged.filter(pl.col("primary_label").is_null())
      .group_by("nepc_review_status", "avpc_review_status").len().sort("len", descending=True))

display(merged.head(20))

In [ ]:
if WRITE_LABELS:
    write_parquet_atomic(merged, OUTPUT_PATH)
    print(f"Wrote {merged.height} rows to {OUTPUT_PATH}")
else:
    print("Skipping write (WRITE_LABELS = False)")